# C4 Requalification Run — T4 GPU

**One-click run.** Just run all cells in order.

**Prerequisites:**
- Runtime → Change runtime type → **T4 GPU** + High-RAM
- Expected time: **~20-30 minutes**

**What it does:**
1. Clones the repo and installs dependencies
2. Runs 607 tests
3. Verifies pre-HRM determinism (120/120 must match)
4. Freezes deterministic packets (immutable, hashed)
5. Runs 7 conformance gates (CPU-only)
6. Runs C4-BRIDGE gate
7. Runs HRM smoke test on GPU
8. Runs full HRM development run (120 tasks × 7 arms = 840 generations)
9. Runs analyzer (quality, gap capture, family CIs)
10. Packages results into a zip for download

## 0. Verify GPU

Make sure you have **T4 GPU** enabled before proceeding.

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")
    print("Stopping here. Please restart runtime after enabling GPU.")
    import sys; sys.exit(0)

## 1. Clone Repository

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/dawsonblock/Daph-ex-research-gate-c2-beir-retrieval.git"
REPO_DIR = "/content/Daph-ex-research-gate-c2-beir-retrieval"

if os.path.exists(REPO_DIR):
    print("Repository exists, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--rebase"], check=False)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"Working directory: {os.getcwd()}")
print(f"Git commit: {commit[:12]}")

## 2. Install Dependencies

transformers >= 5.9.0 is required for the native HRM-Text-1B architecture.

In [ ]:
print("Installing dependencies...")
subprocess.run(["pip", "install", "-q", "transformers>=5.9.0", "huggingface-hub>=0.34"], check=True)
subprocess.run(["pip", "install", "-q", "rank-bm25", "numpy"], check=True)
subprocess.run(["pip", "install", "-q", "pytest"], check=False)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

import transformers
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")
print("Dependencies installed.")

## 3. Run Tests

Verify the repository is GREEN before running. Should be **607 passed, 2 skipped**.

In [ ]:
print("Running test suite (607 tests)...")
result = subprocess.run(["python", "-m", "pytest", "tests/", "-q", "--tb=line", "--timeout=60"],
                       capture_output=True, text=True, timeout=180)
print(result.stdout[-800:] if len(result.stdout) > 800 else result.stdout)
if result.returncode != 0:
    print("\nWARNING: Some tests failed. Check output above.")
    print(result.stderr[-500:])
else:
    print("\nAll tests passed!")

## 4. Pre-HRM Determinism Qualification

Runs 120 tasks with 2 different hash seeds. All packet hashes must be identical.

**If this fails, the pipeline is not deterministic — do not continue to HRM.**

In [ ]:
print("Running determinism qualification (120 tasks, 2 hash seeds)...")
result = subprocess.run(["python", "scripts/c4_determinism_qualification.py",
                         "--tasks", "120", "--seeds", "0,42"],
                       capture_output=True, text=True, timeout=300)
print(result.stdout)
if result.returncode != 0:
    print("\nERROR: Determinism qualification FAILED!")
    print(result.stderr[-500:])
    raise RuntimeError("Pipeline is not deterministic — do not continue")
else:
    print("=== 120/120 identical packet hashes — PASS ===")

## 5. Freeze Deterministic Packets

Generates immutable packet artifacts: `packet.json`, `packet.sha256`, `prompt.txt`, `prompt.sha256`.

HRM will consume these frozen packets. If results change later, you can prove whether the input changed.

In [ ]:
print("Freezing deterministic packets...")
result = subprocess.run(["python", "scripts/c4_freeze_packets.py",
                         "--split", "development"],
                       capture_output=True, text=True, timeout=300)
print(result.stdout)
if result.returncode != 0:
    print("\nERROR: Freeze packets failed!")
    print(result.stderr[-500:])
    raise RuntimeError("Freeze packets failed")
else:
    print("=== Packets frozen successfully ===")

## 6. CPU-Only Dry Run (7 Conformance Gates)

Validates all 7 conformance gates before HRM:
1. No oracle leakage
2. Arm parity
3. Selected IDs in pool
4. Packet budgets
5. Q3 query formulation
6. Merge provenance
7. Causal parity

In [ ]:
print("Running CPU-only dry run (7 conformance gates)...")
result = subprocess.run(["python", "scripts/run_gate_c4.py", "dry-run"],
                       capture_output=True, text=True, timeout=300)
print(result.stdout)
if "ALL VALIDATION GATES PASSED" in result.stdout:
    print("\n=== ALL CONFORMANCE GATES PASSED ===")
else:
    print("\nERROR: Conformance validation failed!")
    print(result.stderr[-1000:])
    raise RuntimeError("Conformance validation failed")

## 7. C4-BRIDGE Gate (No HRM, ~2 seconds)

Bridge qualification gate. Expected: negative result (no runtime bridge mechanism beats the one-pass baseline).

In [ ]:
print("Running C4-BRIDGE gate...")
result = subprocess.run(["python", "scripts/run_gate_c4_bridge.py"],
                       capture_output=True, text=True, timeout=300)
print(result.stdout)
if "BeatsB0=False" in result.stdout:
    print("\n=== C4-BRIDGE NEGATIVE RESULT confirmed (expected) ===")
else:
    print("\nWARNING: Unexpected C4-BRIDGE result (continuing anyway)")

## 8. HRM Smoke Test (3 tasks × 7 arms)

Quick end-to-end test with HRM to verify the model loads on GPU.

Sets `C4_PROTOCOL=v2` for the deterministic, reproducible protocol.

In [ ]:
import os
os.environ["HRM_DEVICE"] = "cuda"
os.environ["HRM_DTYPE"] = "float16"
os.environ["C4_PROTOCOL"] = "v2"

print("Running HRM smoke test (3 tasks × 7 arms)...")
print("This takes ~2-3 minutes on T4...")
result = subprocess.run(["python", "scripts/run_gate_c4.py", "smoke"],
                       capture_output=True, text=True, timeout=600)
print(result.stdout)
if "smoke test complete" in result.stdout:
    print("\n=== Smoke test PASSED ===")
else:
    print("\nERROR: Smoke test failed!")
    print(result.stderr[-1000:])
    raise RuntimeError("Smoke test failed")

## 9. Full HRM Development Run (120 tasks × 7 arms)

**This is the main run.** 840 HRM generations total.

**On T4 GPU: ~15-25 minutes** (vs 3+ hours on CPU)

The run is **resumable** — if Colab disconnects, just re-run this cell and it will pick up where it left off.

In [ ]:
# Check for existing results (resumability)
out_dir = os.path.join(REPO_DIR, "evidence/gate_c4/full/development")
if os.path.exists(out_dir):
    for arm in ["C4_0", "C4_1", "C4_2", "C4_3", "C4_4", "C4_5", "C4_6"]:
        fpath = os.path.join(out_dir, f"{arm}.jsonl")
        if os.path.exists(fpath):
            with open(fpath) as f:
                lines = sum(1 for _ in f if f.strip())
            if lines > 0:
                print(f"  {arm}: {lines}/120 existing results (will resume)")

print("\n=== Starting Full HRM Development Run (C4 Protocol v2) ===")
print("120 tasks × 7 arms = 840 HRM generations")
print("Expected time on T4: ~15-25 minutes")
print()

# Run with real-time output streaming
import subprocess, sys

proc = subprocess.Popen(
    ["python", "scripts/run_gate_c4.py", "full", "--split", "development"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
if proc.returncode != 0:
    print(f"\nERROR: Full run failed with exit code {proc.returncode}")
    raise RuntimeError("Full run failed")
else:
    print("\n=== Full HRM Development Run COMPLETE ===")

## 9b. Diagnostic Arms (Ordering vs Membership Separation)

C4_4 changes TWO things vs C4_3: selector membership AND deterministic packet ordering.

To decompose the effect:
- **C4_3o**: S0 membership + deterministic ordering (ordering-only effect)
- **C4_4m**: S2c membership + pool order (membership-only effect)

Then: `Q(C4_4) - Q(C4_3) = ordering effect + membership effect + interaction`

~5 minutes on T4 (240 additional generations). Optional but recommended.

In [ ]:
print("Running diagnostic arms (C4_3o + C4_4m)...")
print("These decompose Q(C4_4) - Q(C4_3) into ordering vs membership effects.")
print("~5 minutes on T4 (240 additional generations)\n")

proc = subprocess.Popen(
    ["python", "scripts/run_gate_c4.py", "full", "--split", "development",
     "--arms", "C4_3o", "C4_4m"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
if proc.returncode != 0:
    print(f"\nWARNING: Diagnostic arms failed (non-fatal)")
else:
    print("\n=== Diagnostic Arms COMPLETE ===")

## 10. Run Analyzer

Compute all metrics: arm quality, paired deltas, family/cluster/template CIs, task flips, per-regime breakdown, identity stats, selector stats, CSR, role retention, gap capture.

In [ ]:
print("Running C4 analyzer...")
result = subprocess.run(
    ["python", "scripts/analyze_gate_c4.py",
     "--dir", "evidence/gate_c4/full/development",
     "--output", "evidence/gate_c4/full/development/analysis.json"],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.returncode == 0:
    print("\n=== Analysis complete ===")
else:
    print(f"\nAnalyzer error: {result.stderr[-500:]}")

## 11. Composition Diagnostic

Diagnose S2c selection behavior and the canonical regression.

In [ ]:
print("Running composition diagnostic...")
result = subprocess.run(
    ["python", "scripts/diagnose_c4_composition.py"],
    capture_output=True, text=True, timeout=60
)
print(result.stdout)

## 12. Verify Results & Show Summary

Check manifest, hashes, receipt counts, and key metrics.

In [ ]:
import json
from pathlib import Path

out_dir = Path("evidence/gate_c4/full/development")

# Manifest
manifest_path = out_dir / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("=== Manifest ===")
    for key in ["mode", "split", "arm_ids", "task_count", "git_commit",
                "protocol_sha256", "hrm_model_id", "hrm_model_revision",
                "hrm_max_new_tokens", "device", "created_utc"]:
        val = manifest.get(key, "N/A")
        if isinstance(val, str) and len(val) > 20:
            val = val[:20] + "..."
        print(f"  {key}: {val}")

# Receipt counts
print("\n=== Per-arm receipt counts ===")
all_complete = True
for arm_id in ["C4_0", "C4_1", "C4_2", "C4_3", "C4_4", "C4_5", "C4_6"]:
    arm_path = out_dir / f"{arm_id}.jsonl"
    if arm_path.exists():
        lines = [l for l in arm_path.read_text().splitlines() if l.strip()]
        status = "OK" if len(lines) == 120 else "INCOMPLETE"
        if len(lines) != 120:
            all_complete = False
        print(f"  {arm_id}: {len(lines)}/120  [{status}]")
    else:
        all_complete = False
        print(f"  {arm_id}: MISSING")

# Quality summary
analysis_path = out_dir / "analysis.json"
if analysis_path.exists():
    analysis = json.loads(analysis_path.read_text())
    print("\n=== Quality scores ===")
    for arm_id in ["C4_0", "C4_1", "C4_2", "C4_3", "C4_4", "C4_5", "C4_6"]:
        q = analysis.get("arm_quality", {}).get(arm_id)
        if q is not None:
            print(f"  {arm_id}: {q:.4f}")

    delta = analysis.get("primary_delta")
    if delta is not None:
        print(f"\n=== Primary delta (C4_4 - C4_0): {delta:+.4f} ===")
        threshold = 0.15
        if delta >= threshold:
            print(f"  Threshold ({threshold:+.2f}): PASS")
        else:
            print(f"  Threshold ({threshold:+.2f}): FAIL")

    ogc = analysis.get("oracle_gap_capture")
    sgc = analysis.get("selector_gap_capture")
    if ogc is not None:
        print(f"\n  Oracle gap capture:   {ogc:.4f}")
    if sgc is not None:
        print(f"  Selector gap capture: {sgc:.4f}")

# RESULTS.sha256
results_hash = out_dir / "RESULTS.sha256"
if results_hash.exists():
    print(f"\n=== RESULTS.sha256 ===")
    print(results_hash.read_text())

if not all_complete:
    print("\nWARNING: Not all arms have 120 receipts!")
    print("You may need to re-run to complete missing tasks.")

## 13. Download Results

Package everything into a zip and download it.

In [ ]:
import shutil

zip_path = "/content/c4_requalify_results.zip"
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', 'evidence/gate_c4')
size_mb = os.path.getsize(zip_path) / 1e6
print(f"Created: {zip_path}")
print(f"Size: {size_mb:.1f} MB")

try:
    from google.colab import files
    files.download(zip_path)
    print("Download started in browser.")
except ImportError:
    print("(Not in Colab — zip is at /content/c4_requalify_results.zip)")

print("\n=== COMPLETE ===")
print("""
Next steps:
  1. Check primary delta (C4_4 - C4_0) >= +0.15
  2. Check family CI lower bound > 0
  3. Check no canonical/abbreviation regression > 0.05
  4. If development passes → run qualification split
  5. If qualification passes → run OOD split
  6. Gate D decision based on all three splits
""")